# Plantspecies - ChemicalEntity Relation-Wise Merge


## 0. Configuration

In [2]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'databases/'
EVOAGE_DIR = PROC_DIR + '4EvoAGE/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PLANT_CHEMICAL/ALL_PLANT_CHEMICAL.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical Lookups — HMDB, PubChem, DrugBank

In [2]:
# HMDB accession → PubChem CID
HMDB_2_pubchem = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/hmdb/HMDB_withname_pubchem.csv')
HMDB_2_pubchem['PUBCHEM_ID'] = HMDB_2_pubchem['PUBCHEM_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
HMDB_2_pubchemDict = dict(zip(HMDB_2_pubchem['accession'], HMDB_2_pubchem['PUBCHEM_ID']))

print(f"HMDB → PubChem dict: {len(HMDB_2_pubchemDict):,} entries")

HMDB → PubChem dict: 217,920 entries


In [3]:
# PubChem: CID → IUPAC name and SMILES
Pubchem = pd.read_pickle(BASE_DIR + 'data_collection/databases_for_mapping/pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

print(f"PubChem IUPAC dict:  {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict:  119,177,440 entries


In [4]:
# DrugBank: DrugBank ID → drug name
Drugbank = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))

print(f"DrugBank dict:       {len(Drugbank_dict):,} entries")

DrugBank dict:       16,575 entries


### 1b. Disease Lookups — DO, MESH, MedGen

In [5]:
# Disease Ontology: DOID → label and reverse
DO_All_File = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))

print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [6]:
# MedGen: source_id → preferred name; MONDO → MESH CUI mapping
MedGen_CUID_Source_ID = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH    = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict       = dict(zip(MONDO_2_MESH['source_id'],       MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict    = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'],   MONDO_2_MESH['pref_name']))

print(f"MedGen dict:   {len(MedGen_MeshID_name_dict):,} entries")
print(f"MONDO→MESH:    {len(MONDO_2_MESH_dict):,} entries")

MedGen dict:   394,895 entries
MONDO→MESH:    22,703 entries


In [7]:
# MESH xref → DOID mapping (exploded from TSV)
Mesh2DOID = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

print(f"MESH→DOID dict: {len(Mesh2DOID_dict):,} entries")

MESH→DOID dict: 3,687 entries


In [8]:
# MESH combined: ID → Name
MESH = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/MESH_Combined.csv')
MESH_dict      = dict(zip(MESH['ID'], MESH['Name']))
MESH_name_dict = MESH_dict  # alias for clarity

# MESH supplementary: ID → Name
Mesh_supp = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict:      {len(MESH_dict):,} entries")
print(f"MESH supp dict: {len(Mesh_supp_dict):,} entries")

MESH dict:      38,520 entries
MESH supp dict: 324,150 entries


In [9]:
MONDO = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv')
MONDO
MONDO_dict = dict(zip(MONDO['NAME'],MONDO['ID']))
MONDO_dict

{'colorectal Kaposi sarcoma': 'MONDO:0024659',
 'tubular adenoma': 'MONDO:0024660',
 'tubulovillous adenoma': 'MONDO:0024661',
 'colorectal tubulovillous adenoma': 'MONDO:0024662',
 'primary skin meningioma': 'MONDO:0024663',
 'hypertension, pregnancy-induced': 'MONDO:0024664',
 'indeterminate sex and/or pseudohermaphroditism': 'MONDO:0024665',
 'benign epithelial skin neoplasm': 'MONDO:0024666',
 'skin lymphangioma': 'MONDO:0024673',
 'Pancoast syndrome': 'MONDO:0024674',
 'adult kidney Wilms tumor': 'MONDO:0024675',
 'childhood kidney Wilms tumor': 'MONDO:0024676',
 'pancreatic insulinoma': 'MONDO:0024677',
 'Philadelphia-positive myelogenous leukemia': 'MONDO:0024685',
 'tenosynovial giant cell tumor, diffuse type': 'MONDO:0024686',
 'malignant mixed epithelial stromal tumor of the kidney': 'MONDO:0024711',
 'benign synovial neoplasm': 'MONDO:0024715',
 'childhood choroid plexus neoplasm': 'MONDO:0024744',
 'immature teratoma': 'MONDO:0024746',
 'cardiovascular neoplasm': 'MONDO:002

## 2. Load KG Sources

### 2a. ttd 

In [43]:
ttd_Plant_Chemical = pd.read_csv(PROC_DIR + 'ttd/Phytodata_Plantspecies_Chemical.csv')
ttd_Plant_Chemical['kg_type'] = 'Generalised'
ttd_Plant_Chemical['head_id_is'] = ''
ttd_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'
ttd_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
ttd_Plant_Chemical

,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,Curcuma longa (turmeric),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised
1,Solanum torvum (Pea Eggplant),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,glycoalkaloid,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised
2,Rubia cordifolia(Indian Madder),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,124219,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",,Pubchem,https://github.com/asgarhussain/phytochemicals...,Generalised
3,NaN,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised
4,Olea europaea (Olive),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,11652416,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",,Pubchem,https://github.com/asgarhussain/phytochemicals...,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,extract,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised
876,Dracocephalum surmandinum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,864,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,5-(dithiolan-3-yl)pentanoic acid,,Pubchem,https://github.com/asgarhussain/phytochemicals...,Generalised
878,Ferula assa-foetida (Asafoetida),PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,,NaN,https://github.com/asgarhussain/phytochemicals...,Generalised


### 2b. immpat 

In [44]:
immpat_Plant_Chemical = pd.read_csv(PROC_DIR + 'immpat/Phytodata_IMPPAT_PlantSpecies_Chemical.csv')
immpat_Plant_Chemical['kg_type'] = 'Generalised'
immpat_Plant_Chemical['head_id_is'] = ''
immpat_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'
immpat_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
immpat_Plant_Chemical

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,Piper cubeba,PlantSpecies_ChemicalEntity,10992619,PlantSpecies,ChemicalEntity,NaN,"[(1R,2S,5R,6S)-5-benzoyloxy-1,2,6-trihydroxycy...",,Pubchem,IMPPAT,Generalised
1,Clinopodium vulgare,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,NaN,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT,Generalised
2,Cupressus funebris,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,NaN,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT,Generalised
3,Petiveria alliacea,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,NaN,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT,Generalised
4,Torilis leptophylla,PlantSpecies_ChemicalEntity,91714964,PlantSpecies,ChemicalEntity,NaN,(1-methyl-4-prop-1-en-2-ylspiro[4.5]dec-8-en-8...,,Pubchem,IMPPAT,Generalised
...,...,...,...,...,...,...,...,...,...,...,...
118290,Farsetia jacquemontii,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,NaN,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT,Generalised
118291,Lepidium didymum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,NaN,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT,Generalised
118292,Lepidium sativum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,NaN,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT,Generalised
118293,Lepidium virginicum,PlantSpecies_ChemicalEntity,9573945,PlantSpecies,ChemicalEntity,NaN,"[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl] ...",,Pubchem,IMPPAT,Generalised


### 2c. phytohub 

In [45]:
phytohub_Plant_Chemical = pd.read_csv(PROC_DIR + 'phytohub/Phytodata_PhytoHUB_PlantSpecies_Chemical.csv')
phytohub_Plant_Chemical['kg_type'] = 'Generalised'
phytohub_Plant_Chemical['head_id_is'] = ''

phytohub_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'
phytohub_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
phytohub_Plant_Chemical['tail_id_is'] = 'Pubchem'
phytohub_Plant_Chemical

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,Coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,NaN,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",,Pubchem,Phytohub,Generalised
1,Green robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,NaN,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",,Pubchem,Phytohub,Generalised
2,Roasted coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,NaN,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",,Pubchem,Phytohub,Generalised
3,Robusta coffee,PlantSpecies_ChemicalEntity,68103163,PlantSpecies,ChemicalEntity,NaN,"[(1S,4S,12S,13R,16R,17R)-17-methoxy-12-methyl-...",,Pubchem,Phytohub,Generalised
4,Coffee,PlantSpecies_ChemicalEntity,157010316,PlantSpecies,ChemicalEntity,NaN,"(1R,4R,5R,7R,9R,10S,13R,15S)-15-hydroxy-7-[(2S...",,Pubchem,Phytohub,Generalised
...,...,...,...,...,...,...,...,...,...,...,...
1490,Chickpea,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,NaN,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",,Pubchem,Phytohub,Generalised
1491,Red clover,PlantSpecies_ChemicalEntity,5280781,PlantSpecies,ChemicalEntity,NaN,"5-hydroxy-3-(4-methoxyphenyl)-7-[(2S,3R,4S,5S,...",,Pubchem,Phytohub,Generalised
1492,Red clover,PlantSpecies_ChemicalEntity,5281803,PlantSpecies,ChemicalEntity,NaN,"5,7-dihydroxy-3-(3-hydroxy-4-methoxyphenyl)chr...",,Pubchem,Phytohub,Generalised
1493,Red clover,PlantSpecies_ChemicalEntity,5281805,PlantSpecies,ChemicalEntity,NaN,"3-(1,3-benzodioxol-5-yl)-7-hydroxychromen-4-one",,Pubchem,Phytohub,Generalised


### 2d. other sources 

In [46]:
D2_Plant_Chemical = pd.read_csv(PROC_DIR + 'other_sources/D2_Phytodata_PlantSpecies_Chemical_2.csv')
D2_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
D2_Plant_Chemical['kg_type'] = 'Generalised'
D2_Plant_Chemical['head_id_is'] = ''

D2_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'
D2_Plant_Chemical

,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type
0,Origanum vulgare,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
1,Rosmarinus officinalis,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
2,Ocimum basilicum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
3,Thymus vulgaris,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
4,Nigella sativa,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,Nigella sativa,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
77,Mentha pulegium,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
78,Mentha longifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised
79,Mentha piperita,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,338,Species,PlantSpecies,Chemical,ChemicalEntity,NaN,2-hydroxybenzoic acid,,Pubchem,https://github.com/petermr/CEVOpen/blob/master...,Generalised


In [47]:
D3_Plant_Chemical = pd.read_csv(PROC_DIR + 'other_sources/D3_Phytodata_PlantSpecies_Chemical_3.csv')
D3_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
D3_Plant_Chemical['kg_type'] = 'Generalised'
D3_Plant_Chemical['head_id_is'] = ''
D3_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'
D3_Plant_Chemical

,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,tail_id_is,kg_source,kg_type,head_id_is
0,Acokanthera oblongifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,6999980,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"[(3S)-3,7-dimethylocta-1,6-dien-3-yl] acetate",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
1,Acokanthera oblongifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,5281525,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"(3S,6E)-3,7,11-trimethyldodeca-1,6,10-trien-3-ol",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
2,Acokanthera oblongifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,10261,Compound,PlantSpecies,Species,ChemicalEntity,NaN,3-methyl-2-pent-2-enylcyclopent-2-en-1-one,Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
3,Acokanthera oblongifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,133554300,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"3-[(1S,3S,5S,8S,10R,13S,14R,17S)-1,14-dihydrox...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
4,Acokanthera oblongifolia,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,244,Compound,PlantSpecies,Species,ChemicalEntity,NaN,phenylmethanol,Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5625,Xestaea lisianthoides,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,5281653,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"1,8-dihydroxy-2,6-dimethoxyxanthen-9-one",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
5626,Xysmalobium undulatum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,10505,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"3-[(3S,10R,13R,14S)-3,14-dihydroxy-10,13-dimet...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
5627,Xysmalobium undulatum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,321971,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"3-[14-hydroxy-10,13-dimethyl-3-[3,4,5-trihydro...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,
5628,Xysmalobium undulatum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,208007,Compound,PlantSpecies,Species,ChemicalEntity,NaN,"3-[(3S,8R,9S,10R,13R,14S,17R)-3-[(2R,3R,4S,5S,...",Pubchem,https://github.com/alrichardbollans/PhytoChemi...,Generalised,


In [48]:
D4_Plant_Chemical = pd.read_csv(PROC_DIR + 'other_sources/D5_Phytodata_Ayurvedic_PlantSpecies_Chemical.csv')
D4_Plant_Chemical['kg_type'] = 'Generalised'
D4_Plant_Chemical['head_id_is'] = ''
D4_Plant_Chemical.rename(columns={'relation_source': 'kg_source'}, inplace=True)
D4_Plant_Chemical['tail_id_is'] = 'Pubchem'
D4_Plant_Chemical['relation']= 'PlantSpecies_ChemicalEntity'

D4_Plant_Chemical

,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,head_id_is,kg_source,kg_type,tail_id_is
0,Azadirachta indica,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,108058,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"methyl (1S,2R,3R,4R,8R,9S,10R,13R,15R)-2-acety...",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
1,Azadirachta indica,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,12004512,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"[(1S,2R,4S,7S,8S,11R,12R,17R,19R)-7-(furan-3-y...",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
2,Azadirachta indica,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,6437066,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"[(1R,2S,4R,6R,9R,10R,11R,12S,14R,15R,18R)-14-a...",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
3,Azadirachta indica,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,5280343,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"2-(3,4-dihydroxyphenyl)-3,5,7-trihydroxychrome...",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
4,Ocimum sanctum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,3314,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,2-methoxy-4-prop-2-enylphenol,,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
5,Ocimum sanctum,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,2537,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"1,7,7-trimethylbicyclo[2.2.1]heptan-2-one",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
6,Cymbopogon citratus,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,638011,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"(2E)-3,7-dimethylocta-2,6-dienal",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
7,Cymbopogon citratus,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,31253,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"7-methyl-3-methylideneocta-1,6-diene",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
8,Cymbopogon citratus,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,637566,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"(2E)-3,7-dimethylocta-2,6-dien-1-ol",,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem
9,Citrus limon,PlantSpecies_ChemicalEntity,PlantSpecies_ChemicalEntity,22311,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,1-methyl-4-prop-1-en-2-ylcyclohexene,,https://github.com/akshachrya/AyurvedicPlantCl...,Generalised,Pubchem


## 3. Check and Fix Duplicate Columns

`pd.concat` raises `InvalidIndexError` if any DataFrame has duplicate column names.
Inspect here and keep the occurrence with real data.

In [49]:
SOURCE_DFS = [
    ('ttd_Plant_Chemical',ttd_Plant_Chemical),
    ('immpat_Plant_Chemical',   immpat_Plant_Chemical),
    ('phytohub_Plant_Chemical',    phytohub_Plant_Chemical),
    ('D2_Plant_Chemical',                   D2_Plant_Chemical),
    ('D3_Plant_Chemical',                   D3_Plant_Chemical),
    ('D4_Plant_Chemical',                   D4_Plant_Chemical),

]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            print(f"  '{col}':")
            for i, (_, series) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"    occurrence {i} → sample: {series.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicate columns")

[ttd_Plant_Chemical] ✓ no duplicate columns
[immpat_Plant_Chemical] ✓ no duplicate columns
[phytohub_Plant_Chemical] ✓ no duplicate columns
[D2_Plant_Chemical] ✓ no duplicate columns
[D3_Plant_Chemical] ✓ no duplicate columns
[D4_Plant_Chemical] ✓ no duplicate columns


## 4. Align Schemas and Concatenate

In [50]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type','species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 126,397 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Curcuma longa (turmeric),PlantSpecies_ChemicalEntity,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,NaN,Chemical,https://github.com/asgarhussain/phytochemicals...,,NaN,NaN,NaN,Generalised,NaN
1,Solanum torvum (Pea Eggplant),PlantSpecies_ChemicalEntity,glycoalkaloid,PlantSpecies,NaN,Chemical,https://github.com/asgarhussain/phytochemicals...,,NaN,NaN,NaN,Generalised,NaN
2,Rubia cordifolia(Indian Madder),PlantSpecies_ChemicalEntity,124219,PlantSpecies,NaN,Chemical,https://github.com/asgarhussain/phytochemicals...,,Pubchem,NaN,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",Generalised,NaN
3,NaN,PlantSpecies_ChemicalEntity,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,NaN,Chemical,https://github.com/asgarhussain/phytochemicals...,,NaN,NaN,NaN,Generalised,NaN
4,Olea europaea (Olive),PlantSpecies_ChemicalEntity,11652416,PlantSpecies,NaN,Chemical,https://github.com/asgarhussain/phytochemicals...,,Pubchem,NaN,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",Generalised,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
126392,Murraya koenigii,PlantSpecies_ChemicalEntity,5281515,PlantSpecies,NaN,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,,Pubchem,NaN,"(1R,4E,9S)-4,11,11-trimethyl-8-methylidenebicy...",Generalised,NaN
126393,Murraya koenigii,PlantSpecies_ChemicalEntity,14896,PlantSpecies,NaN,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,,Pubchem,NaN,"6,6-dimethyl-2-methylidenebicyclo[3.1.1]heptane",Generalised,NaN
126394,Murraya koenigii,PlantSpecies_ChemicalEntity,31253,PlantSpecies,NaN,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,,Pubchem,NaN,"7-methyl-3-methylideneocta-1,6-diene",Generalised,NaN
126395,Carica papaya,PlantSpecies_ChemicalEntity,9002,PlantSpecies,NaN,Chemical,https://github.com/akshachrya/AyurvedicPlantCl...,,Pubchem,NaN,"4,5-dihydroxybenzene-1,3-disulfonic acid",Generalised,NaN


In [51]:
set(final_df['relation']), set(final_df['head_type']),  set(final_df['tail_type']), set(final_df['head_id_is']),  set(final_df['tail_id_is'])

({'PlantSpecies_ChemicalEntity'},
 {'Compound', 'PlantSpecies', 'Species'},
 {'Chemical', 'ChemicalEntity', 'Species'},
 {''},
 {'Pubchem', nan})

## 5. Standardise Fixed-Value Columns

In [54]:
final_df['head_type'] = 'PlantSpecies'
final_df['tail_type'] = 'ChemicalEntity'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
final_df

Unique relation: {'PlantSpecies_ChemicalEntity'}
Unique head_type: {'PlantSpecies'}
Unique tail_type: {'ChemicalEntity'}
Unique head_id_is: {'Pubchem'}
Unique tail_id_is: {'MESH'}


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Curcuma longa (turmeric),PlantSpecies_ChemicalEntity,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,NaN,ChemicalEntity,https://github.com/asgarhussain/phytochemicals...,Pubchem,MESH,NaN,NaN,Generalised,NaN
1,Solanum torvum (Pea Eggplant),PlantSpecies_ChemicalEntity,glycoalkaloid,PlantSpecies,NaN,ChemicalEntity,https://github.com/asgarhussain/phytochemicals...,Pubchem,MESH,NaN,NaN,Generalised,NaN
2,Rubia cordifolia(Indian Madder),PlantSpecies_ChemicalEntity,124219,PlantSpecies,NaN,ChemicalEntity,https://github.com/asgarhussain/phytochemicals...,Pubchem,MESH,NaN,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",Generalised,NaN
3,nan,PlantSpecies_ChemicalEntity,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,NaN,ChemicalEntity,https://github.com/asgarhussain/phytochemicals...,Pubchem,MESH,NaN,NaN,Generalised,NaN
4,Olea europaea (Olive),PlantSpecies_ChemicalEntity,11652416,PlantSpecies,NaN,ChemicalEntity,https://github.com/asgarhussain/phytochemicals...,Pubchem,MESH,NaN,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",Generalised,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
126392,Murraya koenigii,PlantSpecies_ChemicalEntity,5281515,PlantSpecies,NaN,ChemicalEntity,https://github.com/akshachrya/AyurvedicPlantCl...,Pubchem,MESH,NaN,"(1R,4E,9S)-4,11,11-trimethyl-8-methylidenebicy...",Generalised,NaN
126393,Murraya koenigii,PlantSpecies_ChemicalEntity,14896,PlantSpecies,NaN,ChemicalEntity,https://github.com/akshachrya/AyurvedicPlantCl...,Pubchem,MESH,NaN,"6,6-dimethyl-2-methylidenebicyclo[3.1.1]heptane",Generalised,NaN
126394,Murraya koenigii,PlantSpecies_ChemicalEntity,31253,PlantSpecies,NaN,ChemicalEntity,https://github.com/akshachrya/AyurvedicPlantCl...,Pubchem,MESH,NaN,"7-methyl-3-methylideneocta-1,6-diene",Generalised,NaN
126395,Carica papaya,PlantSpecies_ChemicalEntity,9002,PlantSpecies,NaN,ChemicalEntity,https://github.com/akshachrya/AyurvedicPlantCl...,Pubchem,MESH,NaN,"4,5-dihydroxybenzene-1,3-disulfonic acid",Generalised,NaN


In [55]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

print(f"Before deduplication: {len(final_df):,} rows")
final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
    
})

print(f"After deduplication: {len(final_df_dedup):,} rows")

Before deduplication: 126,397 rows
After deduplication: 125,576 rows


In [57]:
final_df_dedup['species'] = ''
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,36,36
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,125576,0,125576
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,125576,125576,251152


## 12. Save Output

In [60]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 125,576 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PLANT_CHEMICAL/ALL_PLANT_CHEMICAL.csv
